In R I did this to the bulk rnaseq object: 
.libPaths(c(
    "/vsc-hard-mounts/leuven-data/368/YOUR_USERNAME/miniconda3/envs/thesis/lib/R/library",
    .libPaths()
))

library(qs)
cat("qs loaded successfully\n")

bulk <- qread("/path/to/data/PPCG_RNAseq_Version1.2.b_RUVIIIPRPS_K7_10_15.qs")

cat("Class:", class(bulk), "\n")
cat("Dimensions:", dim(bulk), "\n")
cat("First 5 rownames:", rownames(bulk)[1:5], "\n")
cat("First 5 colnames:", colnames(bulk)[1:5], "\n")

library(SummarizedExperiment)

# Get dimensions properly
cat("Genes:", nrow(bulk), "\n")
cat("Samples:", ncol(bulk), "\n")

# Check assay names
cat("Assays:", assayNames(bulk), "\n")

# Check rowData (gene info)
cat("\nrowData columns:", colnames(rowData(bulk)), "\n")
print(head(rowData(bulk)))

# Check colData (sample info)
cat("\ncolData columns:", colnames(colData(bulk)), "\n")
print(head(colData(bulk)))

# Check gene name format
cat("\nFirst 10 rownames:\n")
print(rownames(bulk)[1:10])

# Check if gene symbols can be extracted
gene_names <- gsub(".*\\|\\|", "", rownames(bulk))
cat("\nFirst 10 cleaned gene names:\n")
print(gene_names[1:10])

library(SummarizedExperiment)

# Extract the batch-corrected normalised assay
expr_matrix <- assay(bulk, "RUVIIIPRPS_K_10")

# Clean gene names
rownames(expr_matrix) <- gsub(".*\\|\\|", "", rownames(bulk))

# Remove duplicate gene names keeping first occurrence
expr_matrix <- expr_matrix[!duplicated(rownames(expr_matrix)), ]

cat("Expression matrix dimensions:", dim(expr_matrix), "\n")
cat("First 5 rownames:", rownames(expr_matrix)[1:5], "\n")
cat("First 5 colnames:", colnames(expr_matrix)[1:5], "\n")

# Extract sample metadata
sample_meta <- as.data.frame(colData(bulk)[, c(
    "PPCG_Donor_ID", 
    "PPCG_RNA_Assay_ID",
    "tissue",
    "erg.group",
    "Subtype_Kmeans",
    "purity.scores"
)])

cat("\nSample metadata dimensions:", dim(sample_meta), "\n")
cat("Tissue distribution:\n")
print(table(sample_meta$tissue))

# Filter to tumour samples only
tum_samples <- rownames(sample_meta)[sample_meta$tissue == "TUM"]
cat("\nTumour samples:", length(tum_samples), "\n")

expr_tum <- expr_matrix[, tum_samples]
cat("Tumour expression matrix:", dim(expr_tum), "\n")

# Save
write.csv(
    as.data.frame(expr_tum),
    "/path/to/data/PPCG_bulk_rnaseq_normalised_TUM.csv",
    quote=FALSE
)
write.csv(
    sample_meta,
    "/path/to/data/PPCG_bulk_metadata.csv",
    quote=FALSE
)
cat("Saved expression matrix and metadata\n")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore, rankdata
from scipy.stats import norm as scipy_norm

# ── Load bulk RNA-seq ─────────────────────────────────────────────────
print('Loading bulk RNA-seq...')
bulk = pd.read_csv(
    '/path/to/data/PPCG_bulk_rnaseq_normalised_TUM.csv',
    index_col=0
)
print(f'Bulk shape: {bulk.shape}')  # genes x samples
print(f'First 5 genes: {bulk.index[:5].tolist()}')
print(f'First 5 samples: {bulk.columns[:5].tolist()}')

# ── Load spatial niche DEGs ───────────────────────────────────────────
print('\nLoading spatial DEGs...')
degs = pd.read_csv(
    '/path/to/data/'
    'Cell2location/spatial_mapping/exploration_mean/DEGs/DEGs_joint_alpha_0.9.csv'
)
print(f'DEGs shape: {degs.shape}')
print(f'DEGs columns: {degs.columns.tolist()}')
print(degs.head())

build the niche signatures and score them in the bulk RNA-seq:

In [ ]:
from scipy.sparse import issparse

# ── Check niche distribution in DEGs ─────────────────────────────────
print("DEGs per niche:")
print(degs['niche'].value_counts().sort_index())

# ── Define biological niches only (exclude 0 and 3) ──────────────────
biological_niches = [1, 2, 4, 5, 6]

# ── Get top 200 DEGs per biological niche ────────────────────────────
niche_signatures = {}
for niche in biological_niches:
    niche_degs = degs[degs['niche'] == niche].copy()
    # Sort by score descending and take top 200
    niche_degs = niche_degs.sort_values('score', ascending=False)
    top200 = niche_degs.head(200)['gene'].tolist()
    # Keep only genes present in bulk RNA-seq
    valid = [g for g in top200 if g in bulk.index]
    niche_signatures[niche] = valid
    print(f'Niche {niche}: {len(top200)} DEGs requested, {len(valid)} found in bulk')

import pandas as pd
import numpy as np
from scipy.stats import zscore


# ── 1. Z-Score the Bulk Matrix ────────────────────────────────────────
print('\nZ-scoring bulk RNA-seq matrix across all samples...')
# zscore with axis=1 standardizes each gene (row) across all samples (columns)
bulk_zscored = pd.DataFrame(
    zscore(bulk.values, axis=1), 
    index=bulk.index, 
    columns=bulk.columns
)

# ── 2. Score each sample using the Z-SCORED signature genes ───────────
print('Scoring niche signatures in bulk RNA-seq...')
scores = pd.DataFrame(index=bulk.columns)

for niche, genes in niche_signatures.items():
    if len(genes) == 0:
        print(f'WARNING: Niche {niche} has no genes in bulk — skipping')
        continue
    
    # Calculate the mean of the Z-SCORED expression values
    scores[f'niche_{niche}'] = bulk_zscored.loc[genes].mean(axis=0).values
    print(f'Niche {niche}: scored using {len(genes)} genes')

print(f'\nScores shape: {scores.shape}')
print(scores.describe().round(3))

# ── 3. Extract PPCG patient ID and Aggregate ──────────────────────────
scores['ppcg_id'] = scores.index.str.extract(r'(PPCG\d+)')[0].values

niche_score_cols = [f'niche_{n}' for n in biological_niches]
scores_patient = scores.groupby('ppcg_id')[niche_score_cols].mean()

print(f'\nPatient-level scores shape: {scores_patient.shape}')
print(scores_patient.head())

# ── 4. Save ───────────────────────────────────────────────────────────
scores_patient.to_csv('/path/to/data/niche_signatures_PPCG_patient.csv')
print('Saved patient-level niche scores')

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata
from scipy.stats import norm as scipy_norm
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load metadata and merge ───────────────────────────────────────────
metadata = pd.read_csv(
    '/path/to/data/metadata_common_PPCG.tsv',
    sep='\t'
)
metadata['ppcg_id'] = metadata['ppcg_id'].str.strip()

clinical_cols = ['relapse_ind', 'donor_relapse_interval',
                 'donor_interval_of_last_followup']
metadata_clean = metadata.copy()
for col in clinical_cols:
    metadata_clean[col] = metadata_clean[col].replace(999, np.nan)

# Load niche scores
scores_patient = pd.read_csv(
    '/path/to/data/niche_signatures_PPCG_patient.csv',
    index_col='ppcg_id'
)

# Merge
merged = scores_patient.merge(
    metadata_clean.set_index('ppcg_id')[clinical_cols],
    left_index=True, right_index=True,
    how='inner'
)
print(f"Merged shape: {merged.shape}")

# ── Build survival variables ──────────────────────────────────────────
merged['time'] = np.where(
    merged['relapse_ind'] == 1,
    merged['donor_relapse_interval'],
    merged['donor_interval_of_last_followup']
)
merged['event'] = merged['relapse_ind'].fillna(0).astype(int)
surv = merged.dropna(subset=['time'])
surv = surv[surv['time'] > 0]

print(f"Patients in survival analysis: {len(surv)}")
print(f"Events (relapses): {surv['event'].sum()}")
print(f"Median follow-up: {surv['time'].median():.1f} days")

niche_score_cols = [c for c in surv.columns if c.startswith('niche_')]

# ── Univariate Cox with rank-based normalisation ──────────────────────
results = []
for col in niche_score_cols:
    cox_data = surv[[col, 'time', 'event']].dropna().copy()
    n = len(cox_data)
    ranks = rankdata(cox_data[col])
    cox_data[f'{col}_ranked'] = scipy_norm.ppf((ranks - 0.375) / (n + 0.25))

    cph = CoxPHFitter()
    cph.fit(cox_data[[f'{col}_ranked', 'time', 'event']],
            duration_col='time', event_col='event')

    summary = cph.summary
    results.append({
        'niche': col.replace('niche_', 'Niche '),
        'HR': summary.loc[f'{col}_ranked', 'exp(coef)'],
        'HR_lower': summary.loc[f'{col}_ranked', 'exp(coef) lower 95%'],
        'HR_upper': summary.loc[f'{col}_ranked', 'exp(coef) upper 95%'],
        'p_val': summary.loc[f'{col}_ranked', 'p']
    })

results_df = pd.DataFrame(results)
results_df['padj'] = (results_df['p_val'] * len(results_df)).clip(upper=1.0)
print("\n=== Univariate Cox PH — Spatial niche signatures vs relapse-free survival ===")
print(results_df.round(4).to_string(index=False))
results_df.to_csv('CoxPH_univariate_niche_signatures.csv', index=False)

# ── Forest plot ───────────────────────────────────────────────────────
results_sorted = results_df.sort_values('HR', ascending=True)
labels = results_sorted['niche'].tolist()
hrs    = results_sorted['HR'].tolist()
lower  = results_sorted['HR_lower'].tolist()
upper  = results_sorted['HR_upper'].tolist()
padj   = results_sorted['padj'].tolist()

xerr_low  = [hr - lo for hr, lo in zip(hrs, lower)]
xerr_high = [hi - hr for hr, hi in zip(hrs, upper)]

def sig_label(p):
    if p < 0.001:  return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else:          return 'ns'

sig_labels = [sig_label(p) for p in padj]
colors = []
for hr, p in zip(hrs, padj):
    if p >= 0.05:    colors.append('#999999')
    elif hr > 1:     colors.append('#d62728')
    else:            colors.append('#1f77b4')

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(labels))

ax.errorbar(hrs, y_pos,
            xerr=[xerr_low, xerr_high],
            fmt='o', color='black', ecolor='black',
            elinewidth=1.5, capsize=4, capthick=1.5, markersize=0)

for i, (hr, y, c) in enumerate(zip(hrs, y_pos, colors)):
    ax.scatter(hr, y, color=c, s=80, zorder=5)

for i, (hr, hi, y, sl) in enumerate(zip(hrs, upper, y_pos, sig_labels)):
    ax.text(hi + 0.02, y, f'  {sl}', va='center', fontsize=10)

ax.axvline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlim(0.6, 1.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel(
    'Hazard Ratio (per 1 SD increase in ranked signature score)\nwith 95% confidence interval',
    fontsize=10
)
ax.set_title(
    'Univariate Cox PH — Spatial niche signatures vs relapse-free survival\n'
    'PPCG cohort | Bonferroni-adjusted',
    fontsize=11, fontweight='bold'
)
ax.axvspan(0.6, 1.0, alpha=0.04, color='steelblue')
ax.axvspan(1.0, 1.6, alpha=0.04, color='tomato')
ax.text(0.97, 0.55, 'Risk →', transform=ax.transAxes,
        fontsize=9, color='tomato', ha='right', alpha=0.7)
ax.text(0.03, 0.55, '← Protective', transform=ax.transAxes,
        fontsize=9, color='steelblue', ha='left', alpha=0.7)
ax.text(0.98, 0.02,
        '*** padj<0.001   ** padj<0.01   * padj<0.05   ns not significant',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='grey', alpha=0.8))
plt.tight_layout()
plt.savefig('ForestPlot_CoxPH_niche_signatures.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved forest plot')

In [ ]:
# ── KM curves for significant niches (5 and 6) ───────────────────────
sig_niches = ['niche_5', 'niche_6']
niche_names = {
    'niche_5': 'Niche 5 — Incidental cancer epithelium',
    'niche_6': 'Niche 6 — Luminal prostate epithelium'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, col in enumerate(sig_niches):
    ax = axes[i]
    median_val = surv[col].median()
    high = surv[surv[col] >= median_val]
    low  = surv[surv[col] <  median_val]

    result = logrank_test(
        high['time'], low['time'],
        event_observed_A=high['event'],
        event_observed_B=low['event']
    )
    padj = min(result.p_value * len(sig_niches), 1.0)

    kmf_high = KaplanMeierFitter()
    kmf_low  = KaplanMeierFitter()

    kmf_high.fit(high['time'], high['event'],
                 label=f'High score (n={len(high)})')
    kmf_low.fit(low['time'], low['event'],
                label=f'Low score (n={len(low)})')

    kmf_high.plot_survival_function(ax=ax, ci_show=True, color='steelblue')
    kmf_low.plot_survival_function(ax=ax, ci_show=True, color='tomato')

    ax.set_title(f'{niche_names[col]}\npadj={padj:.4f}', fontsize=10)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Relapse-free survival')
    ax.legend(fontsize=9)

plt.suptitle(
    'Relapse-free survival by spatial niche signature score\n'
    '(median split, PPCG cohort | Bonferroni-adjusted)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('KM_curves_niche_signatures.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved KM curves')